# Dataset2 Part 2: Tiny Aya + AuditKIT
Evaluates its full **160 seed / 10,592 prompt** shard. Both notebooks import the same implementation. No seed crosses parts. Automated guard measurements are not human ground truth.

Open this notebook in Colab, select **Runtime → Change runtime type → GPU**, and run the cells in order. The notebook includes its dataset and shared code and prepares them automatically. No ZIP or separate dataset upload is needed. Each team member runs only their assigned notebook. Keep the same notebook release, revisions and settings for both parts. Default GGUF checkpoints persist in your own Google Drive under `dataset2_eval_gguf/part1` or `part2`. Models and the CUDA server build are reused from Drive cache; GGUF model downloads are public and need no gated HF checkpoint.


## 1. Environment and configuration

In [ ]:
from pathlib import Path
SHARD_ID = 2
RUN_MODE = "full_shard"
BACKEND = "gguf"
REPO_DIR = Path("/content/ez-penta")
USE_DRIVE = True
DATASET_PATH = REPO_DIR / "marcus/dataset2_partial.sqlite3"
MANIFEST_PATH = REPO_DIR / "eval/manifests/dataset2_shards.csv"
OUTPUT_ROOT = Path("/content/drive/MyDrive/ez-penta/dataset2_eval_gguf") if USE_DRIVE else Path("/content/dataset2_eval_gguf")
CACHE_ROOT = Path("/content/drive/MyDrive/ez-penta-cache") if USE_DRIVE else Path("/content/ez-penta-cache")
LOCAL_MODEL_ROOT = Path("/content/models")
AYA_LOCAL_GGUF = None  # optional existing file accessible in Colab/Drive
GUARD_LOCAL_GGUF = None
CONTEXT_SIZE = 8192
GENERATION_SETTINGS = dict(temperature=0.0, top_k=0, top_p=1.0, min_p=0.0, seed=42, max_new_tokens=1024)
GUARD_MAX_TOKENS = 128
BATCH_SIZE = 1
CHECKPOINT_EVERY_RECORD = True
MAX_RETRIES = 2
RETRY_ERRORS = True
RUN_LIKELIHOOD = False  # fp16 diagnostics remain a separate optional workflow
SMOKE_SEED_LIMIT = None  # set RUN_MODE="smoke" and 1 for a complete-seed smoke test


In [ ]:
#@title Automatic setup (dataset, shared code, dependencies)
# The dataset and shared implementation are embedded in this notebook.
# No repository clone, ZIP upload or separate dataset upload is required.
import os, sys, io, base64, hashlib, zipfile, subprocess, importlib, json
from pathlib import Path
REPO_DIR = Path(globals().get("REPO_DIR", "/content/ez-penta"))
ASSETS_SHA256 = '8b82fa33fdffe82fa14e66147b944ad8266ec73ce1872fac287307d24fab9492'
# Bundled bytes live in notebook metadata, not in the code editor.
notebook_file = globals().get("DATASET2_NOTEBOOK_PATH")
if notebook_file is not None:
    notebook_document = json.loads(Path(notebook_file).read_text())
else:
    try:
        from google.colab import _message
    except ImportError as error:
        raise RuntimeError("Open this notebook in Colab, or set DATASET2_NOTEBOOK_PATH to its local .ipynb file.") from error
    print("Reading bundled dataset and code from notebook metadata...", flush=True)
    reply = _message.blocking_request("get_ipynb", timeout_sec=30)
    if isinstance(reply, str):
        reply = json.loads(reply)
    if not isinstance(reply, dict):
        raise RuntimeError("Colab could not read the notebook metadata. Reopen the updated notebook and rerun setup.")
    notebook_document = reply.get("ipynb", reply)
assets = notebook_document.get("metadata", {}).get("dataset2_assets", {})
if assets.get("sha256") != ASSETS_SHA256 or not isinstance(assets.get("payload"), str):
    raise RuntimeError("Bundled metadata is missing or from another release. Open the updated Part 1 or Part 2 notebook file.")
import re
for cell in notebook_document.get("cells", []):
    if cell.get("cell_type") == "code":
        text = cell.get("source", "")
        text = "".join(text) if isinstance(text, list) else text
        if re.search(r"(?m)^\s*[!%]pip\s+install[^\n]*--upgrade[^\n]*(?:huggingface_hub|transformers)", text):
            raise RuntimeError("Remove the extra '!pip install --upgrade huggingface_hub transformers' line from the model-loading cell. It overrides this notebook's tested versions and causes the restart loop. Then rerun setup.")
EMBEDDED_ASSETS = assets["payload"]
asset_bytes = base64.b64decode(EMBEDDED_ASSETS, validate=True)
if hashlib.sha256(asset_bytes).hexdigest() != ASSETS_SHA256:
    raise ValueError("Embedded assets failed integrity verification; reopen the original notebook")
with zipfile.ZipFile(io.BytesIO(asset_bytes)) as archive:
    for member in archive.infolist():
        relative = Path(member.filename)
        if relative.parts[0] != "ez-penta" or ".." in relative.parts or relative.is_absolute():
            raise ValueError("Unsafe embedded asset path")
        destination = REPO_DIR.joinpath(*relative.parts[1:])
        if not destination.resolve().is_relative_to(REPO_DIR.resolve()):
            raise ValueError("Embedded asset path escapes the runtime folder")
        content = archive.read(member)
        if not destination.is_file() or destination.read_bytes() != content:
            destination.parent.mkdir(parents=True, exist_ok=True)
            pending = destination.with_name(destination.name + ".partial")
            pending.write_bytes(content)
            os.replace(pending, destination)
del asset_bytes, EMBEDDED_ASSETS, content, assets, notebook_document
if "reply" in globals():
    del reply
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR.resolve()))
importlib.invalidate_caches()
# Refresh old backend imports without restarting the kernel.
for runner_name in ("aya", "guard_runner"):
    runner = globals().pop(runner_name, None)
    if runner is not None:
        runner.close()
for module_name in ("local_server", "eval.dataset2_runtime", "eval.dataset2_gguf"):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
print("Dataset and shared code extracted; checking dependencies:", REPO_DIR)
from eval import dataset2_environment as setup_environment
importlib.reload(setup_environment)
setup_environment.install_gguf_dependencies(REPO_DIR / "eval/requirements-dataset2-gguf.txt")


## 2. Google Drive and local model cache


In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
print("GGUF models are public. Hugging Face login is not required for this workflow.")
print("Persistent model/build cache:", CACHE_ROOT)


## 3. Dataset loading and shard verification

In [ ]:
# Restore package imports independently of setup-cell kernel state.
import sys, importlib
from pathlib import Path
if "REPO_DIR" not in globals():
    raise RuntimeError("Run the Environment and configuration cell first, then Automatic setup.")
repo_import_path = str(Path(REPO_DIR).resolve())
if not (Path(repo_import_path) / "eval" / "dataset2_data.py").is_file():
    raise RuntimeError("Shared code is not extracted yet. Run the Automatic setup cell near the top, wait for it to finish, then rerun this cell. Runtime > Run all also executes setup in order.")
if repo_import_path not in sys.path:
    sys.path.insert(0, repo_import_path)
importlib.invalidate_caches()
# End import-path recovery.
import json
import pandas as pd
from eval.dataset2_data import load_dataset, verify_manifest, quality_flags, missing_variants
from eval.dataset2_runtime import init_run, atomic_json, stage, make_judge, checkpoint_summary, checkpoint_summary
from eval.dataset2_gguf import GGUFRunner, MODEL_SPECS, GGUF_GENERATION, ensure_cuda_server, prepare_model
from eval.dataset2_analysis import observation_table, summarize, export_bundle
assert BACKEND == "gguf"
assert GENERATION_SETTINGS == dict(temperature=0.0, top_k=0, top_p=1.0, min_p=0.0, seed=42, max_new_tokens=1024)
assert BATCH_SIZE == 1 and GUARD_MAX_TOKENS == 128 and CHECKPOINT_EVERY_RECORD
assert RUN_MODE in ("full_shard", "smoke")
assert (SMOKE_SEED_LIMIT is None) == (RUN_MODE == "full_shard")
manifest = pd.read_csv(MANIFEST_PATH)
dataset_metadata = json.loads(MANIFEST_PATH.with_suffix(".json").read_text())
all_prompts, seeds = load_dataset(DATASET_PATH)
assigned = verify_manifest(all_prompts, seeds, manifest, dataset_metadata, DATASET_PATH)
validated = quality_flags(assigned)
jobs = validated[validated.shard_id == SHARD_ID].copy()
if RUN_MODE == "smoke":
    assert SMOKE_SEED_LIMIT > 0
    selected = sorted(jobs.seed_id.unique())[:SMOKE_SEED_LIMIT]
    jobs = jobs[jobs.seed_id.isin(selected)]
    OUTPUT_ROOT = OUTPUT_ROOT / "smoke"
RUN_DIR = init_run(OUTPUT_ROOT, SHARD_ID, dataset_metadata, RUN_MODE)
rows = jobs.astype(object).where(jobs.notna(), None).to_dict("records")
print(f"Part {SHARD_ID}: {jobs.seed_id.nunique()} seeds, {len(rows)} prompts; mode={RUN_MODE}")
assert RUN_MODE != "full_shard" or (len(rows) == 10592 and jobs.seed_id.nunique() == 160)


## 4. Dataset validation summary

In [ ]:
jobs.to_csv(RUN_DIR / "validation.csv", index=False)
missing = missing_variants(all_prompts, seeds)
missing.to_csv(RUN_DIR / "missing_variants.csv", index=False)
report = {"source_prompts": len(all_prompts), "assigned_prompts": len(jobs),
          "identical_original_twin_rows": int(jobs.identical_twin_flag.sum()),
          "semantic_review_required": int(jobs.semantic_review_required.sum()),
          "generation_excluded": 0, "missing_design_variants": len(missing)}
atomic_json(RUN_DIR / "validation_report.json", report)
display(pd.DataFrame([report]))
display(jobs.groupby(["category", "split"]).size().rename("assigned").to_frame())
# Missing variants are absent generated conditions, not failed Aya evaluations.
# Potential label errors, twin drift and cultural equivalence require human review.


## 5. Tiny Aya model loading

In [ ]:
from eval.dataset2_environment import require_clean_gguf_environment
require_clean_gguf_environment()
build = ensure_cuda_server(cache_root=CACHE_ROOT / "llama-cuda")
aya_path = prepare_model("aya", CACHE_ROOT / "models", LOCAL_MODEL_ROOT, AYA_LOCAL_GGUF)
if "aya" in globals():
    aya.close()
    del aya
aya = GGUFRunner("aya", aya_path, build, RUN_DIR / "logs", context_size=CONTEXT_SIZE)
generation_config = aya.configuration
print("Tiny Aya GGUF Q4_K_M loaded locally; temperature=0.0 (greedy).")


## 6. Response generation

In [ ]:
checkpoint_summary(rows, RUN_DIR / "answers.jsonl", SHARD_ID, generation_config, retry_errors=RETRY_ERRORS)
try:
    answers = stage(rows, RUN_DIR / "answers.jsonl", SHARD_ID, generation_config,
                    aya.answer, max_retries=MAX_RETRIES, retry_errors=RETRY_ERRORS)
finally:
    aya.close()
    del aya
# Each complete or truncated answer is cached. Errors remain visible and retryable.
# Native EOS -> completed; reaching the token cap without EOS -> truncated, excluded from judging.


## 7. Answer checkpoint summary

In [ ]:
display(pd.Series([(r or {}).get("status", "missing") for r in answers.values()]).value_counts().rename("answers").to_frame())
print("Durable checkpoint:", RUN_DIR / "answers.jsonl")


## 8. AuditKIT/Qwen3Guard initialization

In [ ]:
guard_path = prepare_model("guard", CACHE_ROOT / "models", LOCAL_MODEL_ROOT, GUARD_LOCAL_GGUF)
if "guard_runner" in globals():
    guard_runner.close()
    del guard_runner
guard_runner = GGUFRunner("guard", guard_path, build, RUN_DIR / "logs", context_size=CONTEXT_SIZE)
score_answer, judge_config = make_judge(guard_runner)
print("AuditKIT scores saved Aya answers using the local Qwen3Guard GGUF Q8_0 server.")


## 9. Safety and refusal scoring

In [ ]:
checkpoint_summary(rows, RUN_DIR / "scores.jsonl", SHARD_ID, judge_config, answers=answers, retry_errors=RETRY_ERRORS)
try:
    scores = stage(rows, RUN_DIR / "scores.jsonl", SHARD_ID, judge_config,
                   score_answer, answers=answers, max_retries=MAX_RETRIES, retry_errors=RETRY_ERRORS)
finally:
    guard_runner.close()
    del guard_runner
    del score_answer


## 10. Score checkpoint summary

In [ ]:
table = observation_table(rows, answers, scores)
display(table.groupby(["generation_status", "scoring_status", "safety"], dropna=False).size().rename("count").to_frame())
print("Controversial, Unparsed, Missing and Error remain separate from binary safety.")


## 11. Shard-level analysis

In [ ]:
display(summarize(table, ["split"]))
for condition in ["prompt_language", "output_language", "framing", "culture", "category", "expected"]:
    display(summarize(table, ["split", condition]))
# Expected correctness, attack success and over-refusal remain unknown until request
# expectations are validated. Refusal and unsafe response rates are measurements only.
# Heldout results must not guide iterative parameter/judge tuning.


## 12. Manual-inspection samples

In [ ]:
inspection_columns = ["prompt_id", "seed_id", "kind", "culture", "framing", "prompt_language", "output_language", "prompt", "answer", "expected", "safety", "refused", "refusal_heuristic", "raw_guard_outputs", "identical_twin_flag", "human_request_safety", "human_response_safety", "human_refusal", "human_appropriateness", "review_notes", "review_status"]
for label in ["Unsafe", "Safe", "Controversial", "Unparsed", "Missing", "Error"]:
    display(table.loc[table.safety == label, inspection_columns].head(3))
display(table.loc[table.refusal_disagreement.eq(True) | table.identical_twin_flag, inspection_columns].head(10))
# manual_inspection.csv includes every observation; edit human fields externally.


## 13. Optional likelihood-scoring diagnostics

In [ ]:
if RUN_LIKELIHOOD:
    raise RuntimeError("GGUF evaluation does not run HF likelihood diagnostics. Use eval/dataset2_likelihood.py separately with an authorized HF checkpoint and validated candidates; record precision/backend differences.")
print("Likelihood diagnostics disabled. Exact GGUF inputs/output token IDs remain saved for future validated analyses.")


## 14. Final export and completion verification

In [ ]:
metadata = {"shard_id": SHARD_ID, "run_mode": RUN_MODE, "dataset": dataset_metadata,
            "generation": generation_config, "judge": judge_config,
            "analysis_policy": "dataset2-v1-unvalidated-expectations-seed-bootstrap",
            "prompt_ids": sorted(jobs.prompt_id)}
results, coverage = export_bundle(RUN_DIR, table, answers, scores, metadata)
display(pd.DataFrame([coverage]))
display(results["comparisons"])
assert coverage["assigned"] == len(rows) and coverage["duplicate_prompt_ids"] == 0
print("Artifacts:", RUN_DIR)
print("Generation/scoring failures remain explicit. Re-run all cells to resume compatible checkpoints.")
print("Copy/download your entire part directory for merging. The other team member does the same.")


### Merge and inspect
On one machine with this same bundle and both downloaded directories:
```bash
python -m eval.merge_dataset2_results --part1 /path/part1 --part2 /path/part2 --output /path/combined
```
Inspect `combined/coverage.json`, `headline.csv`, `comparisons.csv`, `denominators.csv`, `manual_inspection.csv`, and `results.sqlite3`. Rates are recomputed from observations. Do not average shard rates. The merger rejects mixed model/parser/software configurations, duplicate IDs, missing assignment records and smoke runs. Explicit missing/error observations may still be merged and remain unknown.
